In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import image
from zipfile import ZipFile
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models
from tensorflow.keras import regularizers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = '/content/drive/MyDrive/dog-breed-identification.zip'
with ZipFile(data_path, 'r') as zip:
    zip.extractall()
    print('The data set has been extracted.')

The data set has been extracted.


In [ ]:
labels = pd.read_csv("labels.csv")
BATCH_SIZE = 16
EPOCHS = 20
INPUT_SIZE = 224
NUM_CLASSES = 14  # Adjust the number of breeds

breed_include = ["golden_retriever", "labrador_retriever", "siberian_husky", "chihuahua", "border_collie", "french_bulldog"]
breeds = labels["breed"].value_counts().index[:NUM_CLASSES].tolist()

for breed in breed_include:
  if breed not in breeds:
    breeds.append(breed)


labels = labels[labels["breed"].isin(breeds)].reset_index(drop=True)

NUM_CLASSES = len(breeds)

breed_counts = labels["breed"].value_counts().reset_index()
breed_counts.columns = ["breed", "num_images"]
print(f"Selected {NUM_CLASSES} breeds for training:")
print(breed_counts.to_string(index=False))

Selected 20 breeds for training:
               breed  num_images
  scottish_deerhound         126
         maltese_dog         117
        afghan_hound         116
         entlebucher         115
bernese_mountain_dog         114
            shih-tzu         112
      great_pyrenees         111
          pomeranian         111
             basenji         110
             samoyed         109
            airedale         107
     tibetan_terrier         107
               cairn         106
            leonberg         106
      siberian_husky          95
  labrador_retriever          84
       border_collie          72
           chihuahua          71
      french_bulldog          70
    golden_retriever          67


In [ ]:
from sklearn.model_selection import train_test_split

if not labels['id'].iloc[0].endswith('.jpg'):
    labels['filename'] = labels['id'].astype(str) + '.jpg'
else:
    labels['filename'] = labels['id']  # already has extension

# Add target column for stratification
labels['target'] = labels['breed'].astype('category').cat.codes


Continue Here

In [ ]:
# Stratified split that returns actual DataFrames for convenience
train_df, valid_df = train_test_split(
    labels,
    test_size=0.20,
    stratify=labels['target'],
    random_state=42
)


In [ ]:
# --- Data augmentation/generators ---

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='reflect'
)

valid_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df.reset_index(drop=True),
    directory=os.path.join("/content/train"),
    x_col="filename",
    y_col="breed",
    target_size=(INPUT_SIZE, INPUT_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_generator = valid_datagen.flow_from_dataframe(
    dataframe=valid_df.reset_index(drop=True),
    directory=os.path.join("/content/train"),
    x_col="filename",
    y_col="breed",
    target_size=(INPUT_SIZE, INPUT_SIZE),
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False   # important for consistent evaluation/prediction
)

In [ ]:
# -----------------------------
# Model Definition
# -----------------------------
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(INPUT_SIZE, INPUT_SIZE, 3))

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)

predictions = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

optimizer = Adam(learning_rate=0.00001)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])


In [ ]:
# -----------------------------
# Training
# -----------------------------
history = model.fit(
    train_generator,
    epochs= 10,
    validation_data=valid_generator,
    verbose=1
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Get the true labels and predicted labels for the validation set
valid_labels = valid_generator.classes
valid_pred_probs = model.predict(valid_generator)
valid_preds = np.argmax(valid_pred_probs, axis=1)

# Calculate the confusion matrix
cm = confusion_matrix(valid_labels, valid_preds)

# Plot the confusion matrix as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=valid_generator.class_indices.keys(), yticklabels=valid_generator.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

val_loss, val_acc = model.evaluate(valid_generator, verbose=1)
print(f"Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(epochs, loss, 'o-', label='Training Loss')
plt.plot(epochs, val_loss, 'o-', label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1,2,2)
plt.plot(epochs, acc, 'o-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'o-', label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
val_loss, val_acc = model.evaluate(valid_generator, verbose=1)
print(f"Validation Accuracy: {val_acc*100:.2f}%")


In [ ]:
def display_random_validation_predictions(model, valid_generator, valid_df, num_samples=25):
    """Displays a grid of random validation images with their true and predicted labels."""

    # Get class names from the generator
    class_names = list(valid_generator.class_indices.keys())
    class_names.sort() # Ensure consistent order

    # Select random samples from the validation DataFrame
    sample_df = valid_df.sample(num_samples).reset_index(drop=True)

    plt.figure(figsize=(12, 12))

    for i in range(num_samples):
        row = sample_df.loc[i]
        img_path = os.path.join("/content/train", row['filename']) # Corrected directory to 'train'
        img = image.load_img(img_path, target_size=(INPUT_SIZE, INPUT_SIZE))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) / 255.0

        # Get prediction
        preds = model.predict(img_array, verbose=0)[0]
        predicted_class_index = np.argmax(preds)
        predicted_breed = class_names[predicted_class_index]
        true_breed = row['breed']

        plt.subplot(5, 5, i + 1)
        plt.imshow(img)
        plt.title(f"True: {true_breed}\nPred: {predicted_breed}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Display random predictions
display_random_validation_predictions(model, valid_generator, valid_df)

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_breed(img_path, model, class_indices, top_k=3):

    # Load and preprocess
    img = image.load_img(img_path, target_size=(INPUT_SIZE, INPUT_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension
    img_array = img_array / 255.0  # rescale like training

    # Predict
    preds = model.predict(img_array, verbose=0)[0]

    # Map indices to breed names
    inv_map = {v: k for k, v in class_indices.items()}

    # Sort by probability
    top_indices = preds.argsort()[-top_k:][::-1]
    results = [(inv_map[i], preds[i]) for i in top_indices]

    return results

In [ ]:
# Example usage
img_path = "/content/download.jpg"  # change path
results = predict_breed(img_path, model, train_generator.class_indices, top_k=3)

print("Predictions for:", img_path)
for breed, prob in results:
    print(f"{breed}: {prob:.4f}")

In [ ]:
model.save("dog_breed_inceptionv3.h5")